## 0) 환경 준비 & 구글 드라이브 마운트

In [1]:
# Colab 기준
# 필요한 라이브러리 설치 (이미 설치돼 있으면 건너뜀)
%pip -q install -U transformers accelerate peft bitsandbytes

from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1) 경로/설정 정의

In [2]:
import os, glob, json, requests
import torch
import pandas as pd
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM

# === 사용자가 수정해야 할 부분 ==========================
# LoRA 어댑터가 저장된 드라이브 경로를 지정하세요.
# 예: "/content/drive/MyDrive/llm_outputs/run-001/best"
ADAPTER_DIR = "/content/drive/MyDrive/models/llama3-kor-blossom-8b-lora-best"

# 결과 저장 파일 (드라이브에 저장)
OUT_JSON = "./inference_results_lora.json"

# 데이터셋 URL (필요시 수정)
RAW_URL = "https://raw.githubusercontent.com/beefed-up-geek/HCLT-KACL-2025/main/Korean_Dialogue_Inference/dataset/original_formatted/test.json"

# 베이스 모델 (학습시 사용했던 것과 동일해야 합니다)
MODEL_ID = "MLP-KTLim/llama-3-Korean-Bllossom-8B"
# =======================================================

# ADAPTER_DIR 안에 'best'가 있으면 자동 전환
if os.path.isdir(os.path.join(ADAPTER_DIR, "best")):
    ADAPTER_DIR = os.path.join(ADAPTER_DIR, "best")
print("Adapter candidate path:", ADAPTER_DIR)


Adapter candidate path: /content/drive/MyDrive/models/llama3-kor-blossom-8b-lora-best


## 2) 데이터셋 로드

In [3]:
data = requests.get(RAW_URL).json()
df = pd.DataFrame(data)
print("Loaded samples:", len(df))

Loaded samples: 605


## 3) 베이스 모델 & 토크나이저 재로드

In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)

# 메모리 여유가 충분하면 bf16 + device_map="auto" 권장
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
base_model.eval()

print("Base model loaded on:", base_model.device)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Base model loaded on: cuda:0


## 4) 채팅 템플릿 & 종료 토큰 세팅

In [5]:
SYSTEM_PROMPT = (
    "You are an AI assistant tasked with solving a question based on a two-person conversation. "
    "Carefully read the dialogue, understand the context, and select the most appropriate answer. "
    "당신은 두 사람의 대화를 바탕으로 문제를 해결하는 AI 어시스턴트입니다. "
    "대화를 주의 깊게 읽고 문맥을 이해한 뒤, 가장 적절한 답을 선택하세요."
)

def build_messages(dialogue: str, question: str):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"[Dialogue]\n{dialogue}\n\n[Question]\n{question}\n\nInstruction: Output only the single letter of the correct option (A/B/C)."}
    ]

# eos + eot 종료 토큰 세팅
terminators = [tokenizer.eos_token_id]
try:
    eot_id = tokenizer.convert_tokens_to_ids("<|eot_id|>")
    if isinstance(eot_id, int) and eot_id >= 0:
        terminators.append(eot_id)
except Exception:
    pass

print("Terminators:", terminators)


Terminators: [128009, 128009]


## 5) 드라이브에서 LoRA 어댑터 로드 & 베이스 모델에 장착

In [6]:
from peft import PeftModel

# 어댑터 파일 존재 검증
cfg_ok = os.path.isfile(os.path.join(ADAPTER_DIR, "adapter_config.json"))
wt_ok = len(glob.glob(os.path.join(ADAPTER_DIR, "adapter_model*.safetensors"))) > 0
assert cfg_ok and wt_ok, f"어댑터 파일을 찾을 수 없습니다: {ADAPTER_DIR}"

lora_model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_DIR,
    is_trainable=False,   # 추론 전용
)
lora_model.eval()
print("LoRA adapter loaded.")

MERGE_ADAPTER = True  # 병합을 원하면 True로 변경

if MERGE_ADAPTER:
    merged_model = lora_model.merge_and_unload()
    merged_model.eval()
    use_model = merged_model
    del lora_model
    torch.cuda.empty_cache()
    print("Merged LoRA into base model.")
else:
    use_model = lora_model

print("Using model on:", use_model.device)


LoRA adapter loaded.
Merged LoRA into base model.
Using model on: cuda:0


## 추가) 검증 데이터로 성능 확인 

In [ ]:
import re, numpy as np
import pandas as pd
from tqdm import tqdm

DEV_URL = "https://raw.githubusercontent.com/beefed-up-geek/HCLT-KACL-2025/main/Korean_Dialogue_Inference/dataset/original_formatted/dev.json"
dev_data = requests.get(DEV_URL).json()
df_dev = pd.DataFrame(dev_data)
print("Loaded dev samples:", len(df_dev))

# A/B/C 파싱 유틸
def extract_choice(text: str) -> str:
    if not text:
        return ""
    t = text.strip()
    m = re.search(r"\b([ABC])\b", t)
    if m: return m.group(1)
    m = re.search(r"[정답답안]\s*[:：]\s*([ABC])", t)
    if m: return m.group(1)
    m = re.search(r"[（(]\s*([ABC])\s*[)）]", t)
    if m: return m.group(1)
    for c in "ABC":
        if c in t:
            return c
    return ""

# 카테고리 컬럼명 탐색
possible_cols = ["category", "카테고리", "type", "question_category"]
category_col = next((c for c in possible_cols if c in df_dev.columns), None)

preds, gts, ids, cats, raws = [], [], [], [], []

print("=== dev 데이터셋 평가 시작 ===")
for row in tqdm(df_dev.to_dict(orient="records"), desc="Dev Eval", unit="sample"):
    messages = build_messages(row["dialogue"], row["question"])
    input_ids = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            input_ids,
            max_new_tokens=16,
            eos_token_id=terminators,
            do_sample=False,
            temperature=0.0,
            pad_token_id=tokenizer.eos_token_id
        )

    generated = tokenizer.decode(
        outputs[0][input_ids.shape[-1]:],
        skip_special_tokens=True
    ).strip()

    pred = extract_choice(generated)
    gold = row["answer"].strip()
    cat = row.get(category_col, "UNKNOWN") if category_col is not None else "UNKNOWN"

    preds.append(pred)
    gts.append(gold)
    ids.append(row["id"])
    cats.append(cat)
    raws.append(generated)

# 결과 프레임
res = pd.DataFrame({
    "id": ids,
    "category": cats,
    "gold": gts,
    "pred": preds,
    "correct": [int(p == g) for p, g in zip(preds, gts)],
    "raw": raws,
})

# 카테고리별 집계
cat_order = ["후행사건", "동기", "전제", "반응"]
grp = (
    res.groupby("category", dropna=False)
       .agg(n=("correct", "size"), correct=("correct", "sum"))
       .assign(accuracy=lambda d: d["correct"] / d["n"])
)

ordered_index = [c for c in cat_order if c in grp.index] + [c for c in grp.index if c not in cat_order]
grp = grp.loc[ordered_index]

# 전체 집계
overall = pd.DataFrame({
    "n": [int(res.shape[0])],
    "correct": [int(res["correct"].sum())],
    "accuracy": [res["correct"].mean() if res.shape[0] else np.nan],
}, index=["전체"])

# 최종 표
summary_table = pd.concat([grp, overall], axis=0)

# 보기 좋게 퍼센트 표시
display_table = summary_table.copy()
display_table["accuracy"] = (display_table["accuracy"] * 100).round(2).astype(str) + "%"

print("\n=== dev 카테고리별 및 전체 정확도 ===")
display(display_table)

correct = int(res["correct"].sum())
total = int(res.shape[0])
print(f"\n[dev] Overall Accuracy: {correct}/{total} = {correct/total:.4f}")

## 6) tqdm 포함 추론 루프 (A/B/C → inference_1/2/3) & 드라이브에 저장


In [7]:
# A/B/C → inference_1/2/3
mapping = {"A": "inference_1", "B": "inference_2", "C": "inference_3"}

def pick_choice(text: str):
    t = text.strip().upper()
    # 가장 먼저 등장하는 A/B/C를 채택
    for ch in ("A", "B", "C"):
        if ch in t:
            return ch
    if t in ("A","B","C"):
        return t
    return None

results = []

for row in tqdm(df.to_dict(orient="records"), desc="LoRA Inference", unit="sample"):
    messages = build_messages(row["dialogue"], row["question"])
    input_ids = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(use_model.device)

    with torch.no_grad():
        outputs = use_model.generate(
            input_ids,
            max_new_tokens=16,               # 한 글자면 충분
            eos_token_id=terminators,
            do_sample=False,
            temperature=0.0,
            pad_token_id=tokenizer.eos_token_id
        )

    generated = tokenizer.decode(
        outputs[0][input_ids.shape[-1]:],
        skip_special_tokens=True
    ).strip()

    ch = pick_choice(generated)
    if ch is None:
        tqdm.write(f"[경고] {row['id']} → 응답 파싱 실패: '{generated}'")
        continue

    results.append({
        "id": row["id"],
        "output": mapping[ch]
    })

with open(OUT_JSON, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"총 {len(results)}개 결과 저장 완료 → {OUT_JSON}")


LoRA Inference:   0%|          | 0/605 [00:00<?, ?sample/s]The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
LoRA Inference:   0%|          | 1/605 [00:00<08:28,  1.19sample/s]The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
LoRA Inference:   0%|          | 3/605 [00:01<02:54,  3.45sample/s]The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The fo

총 605개 결과 저장 완료 → ./inference_results_lora.json
